# Transformers & Attention from Scratch

**What you will build**: Every component of a Transformer, from a single attention operation up to a full model that trains on next-token prediction. Then you will load GPT-2 and inspect its learned attention patterns.

**Pedagogy**: Code first, theory second. Each section starts with a working implementation, then asks *why* it works.

---

## 0 | Self-Quiz (Active Recall)

**Answer these from memory before you read any further.** Write your answers in the empty cell below, then compare with the material as you work through the notebook.

1. What fundamental limitation of RNNs does the attention mechanism solve?
2. In the expression `Attention(Q, K, V)`, what do Q, K, and V represent conceptually? Where do they come from?
3. Why do we scale the dot product by `1/sqrt(d_k)` before the softmax?
4. What is the computational complexity of self-attention with respect to sequence length `n`? Why is this a problem?
5. Why do Transformers need positional encodings but RNNs do not?
6. What is the purpose of the residual connection + LayerNorm after each sub-layer?

*Your answers here (double-click to edit):*

1. 
2. 
3. 
4. 
5. 
6. 

---
## 1 | Setup

In [ ]:
# Install dependencies (Colab already has torch, but we pin for reproducibility)
!pip install -q torch matplotlib numpy transformers

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import math

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

---
## 2 | Intuition: Attention as Soft Dictionary Lookup

Forget neural networks for a moment. Imagine you have a **dictionary** with keys and values. A normal dictionary does *exact* matching on the key. Attention does **soft** matching: given a query, it computes similarity to every key, converts similarities to weights (via softmax), and returns a weighted combination of values.

Let's see this in 10 lines of code before we define any class.

In [ ]:
# --- Minimal attention in 10 lines ---
seq_len, d_k = 6, 8  # 6 tokens, embedding dim 8
Q = torch.randn(seq_len, d_k)  # queries
K = torch.randn(seq_len, d_k)  # keys
V = torch.randn(seq_len, d_k)  # values

scores = Q @ K.T / math.sqrt(d_k)  # scaled dot-product similarity
weights = F.softmax(scores, dim=-1)  # normalize to get attention weights
output = weights @ V               # weighted combination of values

print(f"Scores shape:  {scores.shape}")   # (6, 6)
print(f"Weights shape: {weights.shape}")  # (6, 6) -- each row sums to 1
print(f"Output shape:  {output.shape}")   # (6, 8) -- same shape as V
print(f"Row sums of weights: {weights.sum(dim=-1)}")  # all 1.0

In [ ]:
# --- Visualize the attention weights as a heatmap ---
fig, ax = plt.subplots(1, 1, figsize=(6, 5))
im = ax.imshow(weights.detach().numpy(), cmap="Blues", vmin=0, vmax=1)
ax.set_xlabel("Key position")
ax.set_ylabel("Query position")
ax.set_title("Attention Weights (softmax of scaled dot products)")
for i in range(seq_len):
    for j in range(seq_len):
        ax.text(j, i, f"{weights[i, j]:.2f}", ha="center", va="center", fontsize=8)
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

**Key observation**: Each query position produces a *distribution* over all key positions. The output for that position is the weighted average of all values. This is how attention lets every token "look at" every other token in parallel -- no sequential bottleneck like an RNN.

---
## 3 | Single-Head Attention

### The math

Given input $X \in \mathbb{R}^{n \times d_{\text{model}}}$:

$$Q = XW_Q, \quad K = XW_K, \quad V = XW_V$$

$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

Where $W_Q, W_K \in \mathbb{R}^{d_{\text{model}} \times d_k}$ and $W_V \in \mathbb{R}^{d_{\text{model}} \times d_v}$.

### Why scale by $\sqrt{d_k}$?

Without scaling, when $d_k$ is large, the dot products $q \cdot k$ grow in magnitude (variance $\approx d_k$ for unit-variance inputs). Large values push the softmax into saturated regions where gradients are near zero. Dividing by $\sqrt{d_k}$ keeps the variance at $\approx 1$.

In [ ]:
class SingleHeadAttention(nn.Module):
    """Scaled dot-product attention with learnable projections."""

    def __init__(self, d_model: int, d_k: int, d_v: int):
        super().__init__()
        self.d_k = d_k
        self.W_Q = nn.Linear(d_model, d_k, bias=False)
        self.W_K = nn.Linear(d_model, d_k, bias=False)
        self.W_V = nn.Linear(d_model, d_v, bias=False)

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None):
        """
        Args:
            x: (batch, seq_len, d_model)
            mask: (seq_len, seq_len) boolean mask -- True means *ignore* that position
        Returns:
            output: (batch, seq_len, d_v)
            weights: (batch, seq_len, seq_len)
        """
        Q = self.W_Q(x)  # (batch, seq_len, d_k)
        K = self.W_K(x)  # (batch, seq_len, d_k)
        V = self.W_V(x)  # (batch, seq_len, d_v)

        # Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)

        if mask is not None:
            scores = scores.masked_fill(mask, float("-inf"))

        weights = F.softmax(scores, dim=-1)  # (batch, seq_len, seq_len)
        output = torch.matmul(weights, V)    # (batch, seq_len, d_v)

        return output, weights

In [ ]:
# --- Test SingleHeadAttention ---
batch_size, seq_len, d_model, d_k, d_v = 2, 10, 32, 16, 16
x = torch.randn(batch_size, seq_len, d_model)

sha = SingleHeadAttention(d_model, d_k, d_v)
output, weights = sha(x)

print(f"Input shape:   {x.shape}")        # (2, 10, 32)
print(f"Output shape:  {output.shape}")    # (2, 10, 16)
print(f"Weights shape: {weights.shape}")   # (2, 10, 10)
assert output.shape == (batch_size, seq_len, d_v)
assert weights.shape == (batch_size, seq_len, seq_len)
print("All shape assertions passed.")

In [ ]:
# --- Test with causal (autoregressive) mask ---
causal_mask = torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool), diagonal=1)
print("Causal mask (True = blocked):")
print(causal_mask.int())

output_causal, weights_causal = sha(x, mask=causal_mask)
print(f"\nCausal output shape: {output_causal.shape}")

# Verify: upper triangle of attention weights should be 0
upper_triangle = weights_causal[0][torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool), diagonal=1)]
print(f"Max attention weight in masked region: {upper_triangle.max().item():.6f} (should be ~0)")

---
## 4 | Multi-Head Attention

### Why multiple heads?

A single attention head learns a single type of relationship (e.g., "syntactic dependency"). Multiple heads let the model attend to **different relationship types** simultaneously -- one head might capture positional proximity, another coreference, another syntactic structure.

$$\text{MultiHead}(X) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) W_O$$

where $\text{head}_i = \text{Attention}(XW_Q^i, XW_K^i, XW_V^i)$

In practice, we implement this efficiently by projecting to full dimension and then **reshaping** into heads, rather than running `h` separate attention operations.

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-head attention with efficient reshape-based implementation."""

    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"

        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads  # dimension per head

        # Combined projections for efficiency
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def _split_heads(self, x: torch.Tensor) -> torch.Tensor:
        """(batch, seq_len, d_model) -> (batch, n_heads, seq_len, d_k)"""
        batch, seq_len, _ = x.shape
        x = x.view(batch, seq_len, self.n_heads, self.d_k)
        return x.transpose(1, 2)  # (batch, n_heads, seq_len, d_k)

    def _merge_heads(self, x: torch.Tensor) -> torch.Tensor:
        """(batch, n_heads, seq_len, d_k) -> (batch, seq_len, d_model)"""
        batch, _, seq_len, _ = x.shape
        x = x.transpose(1, 2).contiguous()  # (batch, seq_len, n_heads, d_k)
        return x.view(batch, seq_len, self.d_model)

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None):
        """
        Args:
            x: (batch, seq_len, d_model)
            mask: (seq_len, seq_len) boolean mask -- True = ignore
        Returns:
            output: (batch, seq_len, d_model)
            weights: (batch, n_heads, seq_len, seq_len)
        """
        Q = self._split_heads(self.W_Q(x))  # (batch, n_heads, seq_len, d_k)
        K = self._split_heads(self.W_K(x))
        V = self._split_heads(self.W_V(x))

        # Scaled dot-product attention (batched across heads)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)

        if mask is not None:
            # mask shape: (seq_len, seq_len) -> broadcast over batch and heads
            scores = scores.masked_fill(mask.unsqueeze(0).unsqueeze(0), float("-inf"))

        weights = F.softmax(scores, dim=-1)  # (batch, n_heads, seq_len, seq_len)
        attn_output = torch.matmul(weights, V)  # (batch, n_heads, seq_len, d_k)

        # Merge heads and project
        merged = self._merge_heads(attn_output)  # (batch, seq_len, d_model)
        output = self.W_O(merged)                # (batch, seq_len, d_model)

        return output, weights

In [ ]:
# --- Test MultiHeadAttention ---
batch_size, seq_len, d_model, n_heads = 2, 10, 64, 8
x = torch.randn(batch_size, seq_len, d_model)

mha = MultiHeadAttention(d_model, n_heads)
output, weights = mha(x)

print(f"Input shape:   {x.shape}")       # (2, 10, 64)
print(f"Output shape:  {output.shape}")   # (2, 10, 64)  <-- same as input
print(f"Weights shape: {weights.shape}")  # (2, 8, 10, 10)

assert output.shape == (batch_size, seq_len, d_model)
assert weights.shape == (batch_size, n_heads, seq_len, seq_len)
print("All shape assertions passed.")

**Insider Tip:** In frontier lab interviews, you'll be asked to implement multi-head attention from scratch on a whiteboard. The most common mistake candidates make is forgetting the scaling factor or getting the reshape/transpose wrong for splitting heads. Practice this until you can write it in 5 minutes without looking.

In [ ]:
# --- Visualize attention weights from different heads ---
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    w = weights[0, i].detach().numpy()  # head i, first example in batch
    ax.imshow(w, cmap="Blues", vmin=0, vmax=w.max())
    ax.set_title(f"Head {i}")
    ax.set_xlabel("Key pos")
    ax.set_ylabel("Query pos")
plt.suptitle("Attention Patterns Across 8 Heads (random init)", fontsize=14)
plt.tight_layout()
plt.show()

---
## 5 | Positional Encoding

### Why needed?

Self-attention is **permutation-equivariant**: if you shuffle the input tokens, the outputs shuffle in the same way (the attention computation has no notion of position). For language, word order matters ("dog bites man" vs. "man bites dog"), so we must inject positional information.

### Sinusoidal formula (Vaswani et al. 2017)

$$PE_{(pos, 2i)} = \sin\!\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right), \quad PE_{(pos, 2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)$$

This creates a unique "fingerprint" for each position, and (importantly) the relative position between two tokens can be represented as a linear transformation of their encodings.

In [ ]:
class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding (no learnable parameters)."""

    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        # Build the positional encoding table once
        pe = torch.zeros(max_len, d_model)  # (max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)  # (max_len, 1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float) * (-math.log(10000.0) / d_model)
        )  # (d_model/2,)

        pe[:, 0::2] = torch.sin(position * div_term)  # even indices
        pe[:, 1::2] = torch.cos(position * div_term)  # odd indices

        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer("pe", pe)  # not a parameter, but saved with model

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (batch, seq_len, d_model)
        Returns:
            x + positional encoding, same shape
        """
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

In [ ]:
# --- Visualize positional encodings ---
pe_module = PositionalEncoding(d_model=64, max_len=200, dropout=0.0)
# Extract the raw positional encoding (no input needed)
pe_values = pe_module.pe[0, :100, :].numpy()  # first 100 positions

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Full heatmap
im = axes[0].imshow(pe_values, cmap="RdBu", aspect="auto", vmin=-1, vmax=1)
axes[0].set_xlabel("Embedding dimension")
axes[0].set_ylabel("Position")
axes[0].set_title("Sinusoidal Positional Encoding (first 100 positions)")
plt.colorbar(im, ax=axes[0])

# Individual dimension curves
for dim in [0, 1, 4, 5, 20, 21]:
    axes[1].plot(pe_values[:, dim], label=f"dim {dim}")
axes[1].set_xlabel("Position")
axes[1].set_ylabel("Value")
axes[1].set_title("Individual PE Dimensions")
axes[1].legend()

plt.tight_layout()
plt.show()

**Interview insight**: Low-frequency sinusoids (high dimension indices) change slowly across positions and capture *coarse* position. High-frequency sinusoids (low dimension indices) change rapidly and capture *fine* position. This multi-scale representation is similar in spirit to Fourier features.

**Insider Tip:** Modern models use Grouped Query Attention (GQA) and Rotary Position Embeddings (RoPE) instead of sinusoidal. Mention these in interviews -- it shows you know the state of the art. See: 'RoFormer' (Su et al. 2021), 'GQA: Training Generalized Multi-Query Transformer Models' (Ainslie et al. 2023). Nearly all frontier models (Llama 3, Gemma 2, Mistral) use GQA+RoPE as the default architecture.

---
## 6 | Full Transformer Block

A single Transformer block consists of:

1. Multi-Head Attention + Residual + LayerNorm
2. Feed-Forward Network (2-layer MLP) + Residual + LayerNorm

We use **Pre-Norm** architecture (LayerNorm before the sub-layer), which is the default in GPT-2 and most modern LLMs. The original Vaswani paper used Post-Norm.

In [ ]:
class FeedForward(nn.Module):
    """Position-wise feed-forward network with GELU activation."""

    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class TransformerBlock(nn.Module):
    """Single Transformer block with Pre-Norm architecture."""

    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None):
        # Pre-Norm: LayerNorm -> Sub-layer -> Residual
        attn_out, attn_weights = self.attn(self.ln1(x), mask=mask)
        x = x + self.dropout(attn_out)     # residual connection
        x = x + self.ff(self.ln2(x))       # residual connection
        return x, attn_weights

In [ ]:
class MiniTransformer(nn.Module):
    """
    A minimal decoder-only Transformer (GPT-style) for next-token prediction.
    """

    def __init__(
        self,
        vocab_size: int,
        d_model: int = 64,
        n_heads: int = 4,
        n_layers: int = 2,
        d_ff: int = 128,
        max_len: int = 128,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.d_model = d_model
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len, dropout)

        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])

        self.ln_final = nn.LayerNorm(d_model)
        self.output_proj = nn.Linear(d_model, vocab_size)

    def forward(self, input_ids: torch.Tensor):
        """
        Args:
            input_ids: (batch, seq_len) -- token indices
        Returns:
            logits: (batch, seq_len, vocab_size)
        """
        seq_len = input_ids.size(1)

        # Causal mask: prevent attending to future tokens
        causal_mask = torch.triu(
            torch.ones(seq_len, seq_len, dtype=torch.bool, device=input_ids.device),
            diagonal=1
        )

        # Embed and add positional encoding
        x = self.token_emb(input_ids) * math.sqrt(self.d_model)
        x = self.pos_enc(x)

        # Pass through Transformer blocks
        for block in self.blocks:
            x, _ = block(x, mask=causal_mask)

        x = self.ln_final(x)
        logits = self.output_proj(x)  # (batch, seq_len, vocab_size)
        return logits

In [ ]:
# --- Test: verify shapes ---
vocab_size = 100
model = MiniTransformer(vocab_size=vocab_size, d_model=64, n_heads=4, n_layers=2, d_ff=128)
dummy_input = torch.randint(0, vocab_size, (2, 20))  # batch=2, seq_len=20
logits = model(dummy_input)

print(f"Input shape:  {dummy_input.shape}")  # (2, 20)
print(f"Logits shape: {logits.shape}")        # (2, 20, 100)
assert logits.shape == (2, 20, vocab_size)

n_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {n_params:,}")

### Train on a tiny next-token prediction task

We will train our MiniTransformer to memorize a simple repeating pattern. This verifies that gradients flow, attention learns, and the architecture is correct.

In [ ]:
# --- Create a simple repeating-pattern dataset ---
# Pattern: 0 1 2 3 4 5 6 7 8 9 0 1 2 3 ...
vocab_size = 10
seq_len = 32
n_sequences = 200

data = torch.stack([
    torch.tensor([i % vocab_size for i in range(offset, offset + seq_len + 1)])
    for offset in range(n_sequences)
])
# data shape: (200, 33) -- input is [:, :32], target is [:, 1:33]

inputs = data[:, :-1]   # (200, 32)
targets = data[:, 1:]   # (200, 32)

print(f"Sample input:  {inputs[0, :10].tolist()}")
print(f"Sample target: {targets[0, :10].tolist()}")

In [ ]:
# --- Training loop ---
model = MiniTransformer(
    vocab_size=vocab_size, d_model=64, n_heads=4, n_layers=2, d_ff=128, dropout=0.0
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
criterion = nn.CrossEntropyLoss()

losses = []
n_epochs = 100
batch_size = 32

model.train()
for epoch in range(n_epochs):
    # Simple full-batch or mini-batch training
    perm = torch.randperm(n_sequences)
    epoch_loss = 0.0
    n_batches = 0

    for i in range(0, n_sequences, batch_size):
        idx = perm[i:i + batch_size]
        x_batch = inputs[idx].to(device)
        y_batch = targets[idx].to(device)

        logits = model(x_batch)  # (batch, seq_len, vocab_size)
        loss = criterion(logits.reshape(-1, vocab_size), y_batch.reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        n_batches += 1

    avg_loss = epoch_loss / n_batches
    losses.append(avg_loss)
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d}/{n_epochs}: loss = {avg_loss:.4f}")

# Plot training loss
plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel("Epoch")
plt.ylabel("Cross-Entropy Loss")
plt.title("MiniTransformer Training on Repeating Pattern")
plt.grid(True)
plt.show()

In [ ]:
# --- Verify: model should predict the next number in the cycle ---
model.eval()
test_input = torch.tensor([[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 0, 1]], device=device)
with torch.no_grad():
    test_logits = model(test_input)
    predictions = test_logits.argmax(dim=-1)

print("Input:       ", test_input[0].tolist())
print("Predictions: ", predictions[0].tolist())
print("Expected:    ", [1, 2, 3, 4, 5, 6, 7, 8, 9, 0, 1, 2])

---
## 7 | Loading Pretrained GPT-2 & Inspecting Attention

Now let's load a *real* pretrained Transformer and see what attention patterns it has learned.

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
gpt2 = GPT2LMHeadModel.from_pretrained("gpt2", output_attentions=True)
gpt2.eval()

print(f"GPT-2 parameters: {sum(p.numel() for p in gpt2.parameters()):,}")
print(f"Layers: {gpt2.config.n_layer}, Heads: {gpt2.config.n_head}, d_model: {gpt2.config.n_embd}")

In [ ]:
# --- Generate text with GPT-2 ---
prompt = "The theory of attention in neural networks"
input_ids = tokenizer.encode(prompt, return_tensors="pt")

with torch.no_grad():
    output = gpt2.generate(
        input_ids,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.8,
        top_k=50,
        pad_token_id=tokenizer.eos_token_id,
    )

generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
print("Generated text:")
print(generated_text)

In [ ]:
# --- Inspect attention patterns on a specific sentence ---
sentence = "The cat sat on the mat because it was soft"
input_ids = tokenizer.encode(sentence, return_tensors="pt")
tokens = tokenizer.convert_ids_to_tokens(input_ids[0])

with torch.no_grad():
    outputs = gpt2(input_ids)
    # outputs.attentions is a tuple of (n_layers,) each (batch, n_heads, seq_len, seq_len)
    all_attentions = outputs.attentions

print(f"Number of layers: {len(all_attentions)}")
print(f"Attention shape per layer: {all_attentions[0].shape}")
print(f"Tokens: {tokens}")

In [ ]:
# --- Plot attention for selected layers and heads ---
layers_to_plot = [0, 5, 11]  # first, middle, last
heads_to_plot = [0, 4, 8, 11]

fig, axes = plt.subplots(len(layers_to_plot), len(heads_to_plot), figsize=(20, 14))

for row, layer_idx in enumerate(layers_to_plot):
    for col, head_idx in enumerate(heads_to_plot):
        attn = all_attentions[layer_idx][0, head_idx].numpy()
        ax = axes[row, col]
        ax.imshow(attn, cmap="Blues", vmin=0)
        ax.set_xticks(range(len(tokens)))
        ax.set_xticklabels(tokens, rotation=45, ha="right", fontsize=7)
        ax.set_yticks(range(len(tokens)))
        ax.set_yticklabels(tokens, fontsize=7)
        ax.set_title(f"Layer {layer_idx}, Head {head_idx}", fontsize=9)

plt.suptitle(f'GPT-2 Attention Patterns: "{sentence}"', fontsize=14)
plt.tight_layout()
plt.show()

**What to look for in the attention plots**:
- **Diagonal patterns**: token attending to itself or its immediate predecessor
- **Columnar patterns**: all tokens attending to a specific "anchor" token (often the first token or punctuation)
- **Pronoun resolution**: does "it" attend to "mat" or "cat"? This is a classic test of coreference
- **Early layers** tend to show more local/positional patterns; **later layers** show more semantic patterns

**Insider Tip:** Flash Attention (Dao et al. 2022, 2023) is the standard for efficient attention computation in production. If asked about attention optimization, this is the first thing to mention. Flash Attention fuses the attention computation into a single GPU kernel, reducing memory from O(n^2) to O(n) by never materializing the full attention matrix. Flash Attention 2 and 3 further optimize with better work partitioning and hardware-aware scheduling. Every frontier lab uses it.

---
## 8 | "Why Does This Work?" -- Elaborative Interrogation

For each question below, write a 2-3 sentence answer. This is the most effective study technique for deep understanding.

1. **Why does attention outperform RNNs on long sequences?** Think about gradient flow, parallelism, and the "path length" between distant tokens.

2. **Why is the softmax necessary?** What would happen if we just used the raw dot-product scores to weight the values?

3. **Why do we need the output projection $W_O$ in multi-head attention?** What would the output look like without it?

4. **Why Pre-Norm instead of Post-Norm?** GPT-2 and most modern LLMs use Pre-Norm. What training stability properties does this give?

5. **Why GELU instead of ReLU in the feed-forward network?** Think about the smoothness of gradients near zero.

6. **Why does the original Transformer scale embeddings by $\sqrt{d_{\text{model}}}$ before adding positional encodings?** (Hint: consider the relative magnitudes.)

7. **What is the relationship between attention and kernel methods?** (Advanced: think about the softmax kernel $K(q,k) = \exp(q \cdot k)$.)

8. **Why does GPT-2 use learned positional embeddings instead of sinusoidal?** Does it matter empirically?

*Your answers here:*

1. 
2. 
3. 
4. 
5. 
6. 
7. 
8. 

---
## Interview Question Bank: Transformers & Attention

*These are representative questions modeled on those at Anthropic, OpenAI, DeepMind, and Meta FAIR for senior/principal ML roles. Representative interview-style questions (synthetic, not sourced from real debriefs).*

---

### Question 1: "Implement multi-head attention from scratch."

**What we're testing:** Core ML engineering fluency. This is THE most common ML coding question at frontier labs. If you can't do this, you won't pass the technical screen.

**Good answer:** Correct implementation of Q/K/V projections, scaled dot-product attention, concatenation and output projection, causal masking -- all completed in under 15 minutes. Mentions the scaling factor $\frac{1}{\sqrt{d_k}}$ and explains why (prevents softmax saturation).

**Great answer (Principal-level):** Implements it cleanly AND then discusses modern architectural choices unprompted: Grouped Query Attention (GQA) as used in Llama 2/3 for KV-cache efficiency, Rotary Position Embeddings (RoPE) instead of sinusoidal/learned positions, Flash Attention for IO-aware memory-efficient computation. Mentions that real models use Pre-LayerNorm (not Post-LayerNorm) and explains the training stability difference.

**Red flag:** Takes more than 20 minutes. Confuses the roles of Q, K, V. Can't explain why we scale by $\sqrt{d_k}$. Doesn't know what causal masking is for.

**Follow-up 1:** "Now make it memory-efficient for a 128K context window." (Tests knowledge of Flash Attention -- the key insight is that standard attention materializes the full $n \times n$ attention matrix in HBM, while Flash Attention tiles the computation to keep everything in SRAM, reducing memory from $O(n^2)$ to $O(n)$ and actually running faster despite more FLOPs.)

**Follow-up 2:** "How would you modify this attention mechanism for a Mixture-of-Experts model?" (Tests understanding that MoE applies to FFN layers, not attention -- but top-k routing decisions interact with attention patterns. Strong candidates discuss expert load balancing and the auxiliary loss.)

---

### Question 2: "What is the computational complexity of attention? How do you reduce it?"

**What we're testing:** Algorithmic thinking and awareness of practical bottlenecks.

**Good answer:** $O(n^2 d)$ for sequence length $n$ and dimension $d$. The quadratic term comes from every token attending to every other token. Mentions that this is why long-context models are expensive.

**Great answer (Principal-level):** Distinguishes between FLOP complexity and memory complexity. Standard attention is $O(n^2 d)$ FLOPs and $O(n^2)$ memory. Flash Attention doesn't change the FLOPs but reduces memory to $O(n)$ by never materializing the full attention matrix -- it's an IO-aware algorithm that exploits the GPU memory hierarchy (SRAM vs HBM). Also discusses: sparse attention (BigBird, Longformer), linear attention attempts (they mostly didn't work well in practice), Ring Attention for distributing long sequences across multiple GPUs, and why most "efficient attention" papers from 2020-2021 lost to just scaling with Flash Attention.

**Red flag:** Says "attention is O(n^2)" without specifying what the complexity is measured in (time vs memory vs both). Claims linear attention works just as well. Doesn't know Flash Attention.

**Follow-up:** "If I give you 8 GPUs and a 1M token context, how do you run inference?" (Tests Ring Attention / sequence parallelism understanding.)

---

### Question 3: "Explain positional encoding. Why not just use learned positions?"

**What we're testing:** Depth of understanding of a fundamental design choice and awareness of the field's evolution.

**Good answer:** Transformers are permutation-equivariant without positional information, so we need to inject position somehow. Sinusoidal encodings (original Transformer) use fixed sinusoidal functions of different frequencies. Learned embeddings (GPT-2, BERT) treat positions as a lookup table.

**Great answer (Principal-level):** Explains the full evolution: sinusoidal (Vaswani) -> learned absolute (BERT/GPT-2) -> relative (T5, ALiBi) -> rotary (RoPE, used in virtually all modern LLMs). Explains WHY RoPE won: it encodes relative position through rotation matrices applied to Q and K, supports length extrapolation better than learned positions, and is compatible with KV-caching. Discusses NTK-aware RoPE scaling and YaRN for extending context windows beyond training length. Knows that ALiBi (Attention with Linear Biases) is simpler but RoPE generalizes better empirically.

**Red flag:** Only knows sinusoidal encoding. Can't explain why position information is needed at all. Doesn't know RoPE.

**Follow-up:** "You've trained a model with 8K context and want to extend it to 128K. What are your options and trade-offs?" (Tests RoPE scaling, continued pretraining, YaRN, and the compute/quality trade-off of each approach.)

---
## Production Implementation Notes: Attention & Transformers at Scale

*What the production version looks like vs. what you just implemented in this notebook.*

### The Textbook vs. Reality Gap

| Component | Textbook Version (this notebook) | Production Version (frontier labs) |
|-----------|--------------------------------|-----------------------------------|
| **Attention** | Multi-Head Attention (MHA) | Grouped Query Attention (GQA) -- Llama 2/3, Gemma, Mistral |
| **Position encoding** | Sinusoidal or learned absolute | RoPE (Rotary Position Embeddings) -- nearly universal |
| **Layer norm** | Pre-Norm | Pre-LayerNorm (or RMSNorm for efficiency) |
| **Attention computation** | Naive $O(n^2)$ memory | Flash Attention 2/3 -- IO-aware, tiled computation |
| **Activation function** | ReLU | SwiGLU (PaLM, Llama), GeGLU (T5 v1.1, Gemma) |
| **Precision** | float32 | BFloat16 training, FP8 for some operations |

### Scale Numbers You Should Know

- **Training a frontier model (e.g., Llama 3 405B):** 10,000-16,000 H100 GPUs, 3-4 months of continuous training, ~$100M+ in compute
- **Pretraining data:** 10-15 trillion tokens (deduplicated, quality-filtered from petabytes of raw data)
- **Single forward pass (70B model):** ~140 GFLOPs per token (≈ 2 × params) -- FLOPs measures work, FLOP/s measures hardware rate; ~140GB of memory for parameters alone (in BF16)
- **KV-cache at 128K context (70B):** Can exceed 100GB per request -- this is why GQA matters (reduces KV-cache by 4-8x vs MHA)
- **Inference serving:** One 70B model needs 4-8 H100 GPUs just for inference, serving thousands of concurrent requests requires sophisticated batching (vLLM, TensorRT-LLM)

### Engineering Challenges That Don't Appear in Papers

1. **Training instability at scale:** Loss spikes that don't happen at small scale. Teams maintain "spike detection" systems that automatically checkpoint and roll back. Llama 3 405B pre-training saw 466 job interruptions over a 54-day snapshot -- 419 of them unexpected, with ~78% of the unexpected interruptions hardware-related.
2. **Data pipeline bottleneck:** At 10K+ GPU scale, the data loader must sustain TB/s of throughput. This is a serious distributed systems problem.
3. **GPU failures:** With 10K GPUs running for months, hardware failures happen multiple times per day. Elastic training, rapid checkpointing, and automatic recovery are critical infrastructure.
4. **Numerical precision:** BF16 training requires careful loss scaling. Some operations (attention softmax, layer norm) need higher precision. Mixed precision strategies are an art.
5. **Parallelism stack:** Real training uses 4 types of parallelism simultaneously: data parallel (FSDP/ZeRO), tensor parallel, pipeline parallel, and sequence parallel. Getting the communication topology right for your cluster is a multi-week engineering effort.

### Monitoring & Evaluation in Production

- **Loss curves:** Watched in real-time by on-call engineers. Any deviation triggers investigation.
- **Gradient norms:** Per-layer gradient norms detect instability before it becomes a loss spike.
- **Activation statistics:** Monitor for dead neurons, saturated activations, or anomalous patterns.
- **Downstream evals:** Automated evaluation on benchmark suites (MMLU, HumanEval, etc.) at regular checkpoint intervals.
- **Throughput monitoring:** MFU (Model FLOPs Utilization) should stay above 40-50%. Drops indicate communication bottlenecks or hardware issues.

---
## How This Gets Tested in Interviews

### Where Attention Questions Appear

| Company | Round | Format | Depth |
|---------|-------|--------|-------|
| **Anthropic** | Technical screen + onsite | Coding (implement attention) + discussion (architectural choices) | Deep -- expects Flash Attention, GQA, RoPE knowledge |
| **OpenAI** | Technical screen | Coding (implement from scratch in 20 min) | Moderate to deep -- focus on correctness + efficiency |
| **DeepMind** | Onsite (ML fundamentals) | Whiteboard/coding + mathematical derivation | Very deep -- may ask you to derive backprop through attention |
| **Meta FAIR** | Phone screen + onsite | Coding + system design | Moderate -- implementation correctness, then scale discussion |
| **Google Brain/DeepMind** | Onsite | Mixed coding + discussion | Expects mathematical rigor (prove properties of attention) |

### Time Expectations

- **"Implement multi-head attention from scratch"**: 15-20 minutes. This is a timed exercise. Practice until you can do it in 12 minutes to leave buffer for edge cases and discussion.
- **"Explain the complexity and how to reduce it"**: 5-10 minute discussion. You should be able to talk fluidly about this without notes.
- **"Design a long-context attention system"**: 30-45 minute system design. This is a senior/principal-level question that combines attention mechanics with distributed systems.

### Senior vs. Principal Expectations

**senior ML engineer:**
- Implement MHA correctly and quickly
- Explain complexity trade-offs
- Know Flash Attention exists and why it matters
- Understand causal masking, KV-caching basics

**principal ML engineer:**
- All of the above, plus:
- Design decisions: when to use GQA vs MQA vs MHA (and the KV-cache memory math)
- RoPE internals: how rotation matrices encode relative position, NTK-aware scaling
- Flash Attention: explain the tiling algorithm, SRAM vs HBM trade-off, backward pass differences
- System design: how to serve 128K context models, sequence parallelism, Ring Attention
- Historical context: why previous "efficient attention" approaches (linear, sparse) didn't replace standard attention + Flash Attention
- Open questions: what are the current limitations and active research directions

### Preparation Checklist

- [ ] Implement MHA from scratch in under 15 minutes (time yourself)
- [ ] Implement causal masking and explain why it's needed for autoregressive generation
- [ ] Explain Flash Attention at a whiteboard (the tiling, the online softmax trick)
- [ ] Know the GQA paper: how many KV heads does Llama 3 use? (8 KV heads for 70B)
- [ ] Explain RoPE: what does the rotation matrix look like? Why does it encode relative position?
- [ ] Be ready for: "What would you change about the Transformer architecture?" (have a thoughtful, opinionated answer)

---
## 9 | Flashcard Summary (Anki Export Format)

| # | Question | Answer |
|---|----------|--------|
| 1 | What is the core equation for scaled dot-product attention? | $\text{Attention}(Q,K,V) = \text{softmax}(QK^T / \sqrt{d_k})\, V$ |
| 2 | Where do Q, K, V come from? | They are linear projections of the input: $Q = XW_Q$, $K = XW_K$, $V = XW_V$ |
| 3 | Why scale by $\sqrt{d_k}$? | Without scaling, large $d_k$ causes dot products to have high variance, pushing softmax into saturated regions with near-zero gradients |
| 4 | What is the complexity of self-attention w.r.t. sequence length? | $O(n^2 d)$ -- quadratic in sequence length due to the $n \times n$ attention matrix |
| 5 | What problem does multi-head attention solve over single-head? | Multiple heads let the model attend to different types of relationships simultaneously (syntactic, semantic, positional) |
| 6 | How is multi-head attention implemented efficiently? | Project to full $d_{\text{model}}$, reshape into $(\text{batch}, n_{\text{heads}}, \text{seq}, d_k)$, compute attention in parallel, merge, project with $W_O$ |
| 7 | Why do Transformers need positional encodings? | Self-attention is permutation-equivariant -- without PE, the model cannot distinguish word order |
| 8 | What are the two components of a Transformer block? | (1) Multi-Head Attention + Residual + LayerNorm, (2) Feed-Forward Network + Residual + LayerNorm |
| 9 | What is the difference between Pre-Norm and Post-Norm? | Pre-Norm applies LayerNorm *before* the sub-layer (better training stability). Post-Norm applies it *after* (original paper). |
| 10 | What does a causal mask do? | Prevents each position from attending to future positions, enforcing autoregressive generation. Implemented by setting future scores to $-\infty$ before softmax. |
| 11 | What is the role of the feed-forward network in each Transformer block? | Applies a non-linear transformation independently to each position. Often interpreted as a "memory" that stores factual knowledge. |
| 12 | Why multiply embeddings by $\sqrt{d_{\text{model}}}$ before adding positional encodings? | This is init-dependent: in the original Transformer (shared embedding/output weights, Xavier-style init with variance $\sim 1/d$), scaling by $\sqrt{d_{\text{model}}}$ restores roughly unit scale to match the positional encodings. With `nn.Embedding`'s default $\mathcal{N}(0,1)$ init (as in this notebook's code), that scaling would *inflate* the variance instead -- most modern LLMs drop it. |

---
## 10 | Paper Reading Guide

### "Attention Is All You Need" -- Vaswani et al. (2017)

**Link**: [https://arxiv.org/abs/1706.03762](https://arxiv.org/abs/1706.03762)

### 3-Sentence Summary

The paper introduces the Transformer architecture, which replaces recurrence and convolutions entirely with self-attention mechanisms for sequence transduction tasks. The key innovation is multi-head scaled dot-product attention, which allows the model to jointly attend to information from different representation subspaces at different positions. The architecture achieved state-of-the-art results on machine translation (WMT 2014 English-to-German and English-to-French) while being more parallelizable and requiring significantly less training time than RNN-based models.

### Reading Guide: Key Sections

| Section | What to focus on | Time |
|---------|-----------------|------|
| Abstract + Intro | The motivation: why replace RNNs? What are the claimed benefits? | 5 min |
| Section 3.1 (Encoder-Decoder) | Overall architecture. Understand the diagram in Figure 1 deeply. | 10 min |
| Section 3.2 (Attention) | **Most important section.** Equations 1-3. Understand scaled dot-product and multi-head attention. | 20 min |
| Section 3.3 (Position-wise FFN) | Simple but important: why is this needed in addition to attention? | 5 min |
| Section 3.5 (Positional Encoding) | Sinusoidal encoding formula. Why this particular choice? | 10 min |
| Section 5.4 (Regularization) | Dropout, label smoothing -- practical training details | 5 min |
| Table 3 (Ablations) | **Critical for interviews.** How does each design choice affect performance? | 10 min |

### Interview-Relevant Questions from This Paper

1. Walk me through the Transformer architecture from input to output.
2. Why is self-attention $O(n^2 d)$ and how might you reduce this?
3. The paper compares self-attention to RNNs and CNNs (Table 1). Reproduce this analysis.
4. What role does label smoothing play in the training procedure?
5. How would you modify this architecture for a decoder-only (GPT-style) model?

### Related Papers to Read Next

- **GPT-2** (Radford et al. 2019): Decoder-only Transformer for language modeling
- **BERT** (Devlin et al. 2019): Encoder-only Transformer with masked language modeling
- **FlashAttention** (Dao et al. 2022): IO-aware exact attention algorithm -- key systems paper
- **FlashAttention-2** (Dao 2023): Better work partitioning and parallelism for 2x speedup over FA1
- **RoPE** (Su et al. 2021): Rotary position embeddings -- used in LLaMA, modern standard
- **GQA** (Ainslie et al. 2023): Grouped Query Attention -- the KV-cache-efficient attention variant used in Llama 2/3, Gemma, Mistral
- **DeepSeek-V2** (DeepSeek 2024): Multi-head Latent Attention (MLA) -- novel KV compression that has already shipped in DeepSeek-V2/V3/R1; GQA remains the default in most other open models
- **Llama 3** (Meta 2024): State-of-the-art open model showcasing modern Transformer design choices (GQA + RoPE + SwiGLU)